In [26]:
import pandas as pd 
import numpy as np
import requests
import pyarrow.parquet as pq
import io
import statsmodels.api as sm

## CARGA DE DATOS

In [27]:


def leer_parquet_desde_url(url):
    resp = requests.get(url)
    resp.raise_for_status()  # Asegura que la descarga fue exitosa
    buf = io.BytesIO(resp.content)
    table = pq.read_table(buf)
    return table.to_pandas()

url_bodegas = "https://raw.githubusercontent.com/Carlos2935/Proyecto_aplicado/master/bodegas.parquet"
url_eventos = "https://raw.githubusercontent.com/Carlos2935/Proyecto_aplicado/master/eventos.parquet"
url_productos = "https://raw.githubusercontent.com/Carlos2935/Proyecto_aplicado/master/productos.parquet"

bodegas = leer_parquet_desde_url(url_bodegas)
eventos = leer_parquet_desde_url(url_eventos)
productos = leer_parquet_desde_url(url_productos)

In [28]:
# Imprimir las columnas de cada DataFrame
print("Columnas de bodegas:")
print(bodegas.columns.tolist())  # .tolist() para verlas como lista

print("\nColumnas de eventos:")
print(eventos.columns.tolist())

print("\nColumnas de productos:")
print(productos.columns.tolist())

Columnas de bodegas:
['id_bodega', 'ciudad', 'nombre_bodega', 'tipo', 'region', 'pais', 'antiguedad_dias']

Columnas de eventos:
['fecha', 'id_producto', 'id_bodega', 'tipo_evento', 'cantidad', 'cantidad_2', 'precio', 'precio_total', 'costo', 'col_extra_1', 'col_extra_2', 'col_extra_3', 'id_ticket']

Columnas de productos:
['id_producto', 'desc_producto', 'talla', 'color', 'marca', 'genero', 'categoria', 'subcategoria', 'patron', 'largo', 'temporada', 'tipo', 'material']


In [29]:
df_elasticidad =eventos.merge(
    productos,
    on="id_producto",
    how="left"
)

df_elasticidad = df_elasticidad.merge(
    bodegas,
    on="id_bodega",
    how="left"
)
df_elasticidad.columns

Index(['fecha', 'id_producto', 'id_bodega', 'tipo_evento', 'cantidad',
       'cantidad_2', 'precio', 'precio_total', 'costo', 'col_extra_1',
       'col_extra_2', 'col_extra_3', 'id_ticket', 'desc_producto', 'talla',
       'color', 'marca', 'genero', 'categoria', 'subcategoria', 'patron',
       'largo', 'temporada', 'tipo_x', 'material', 'ciudad', 'nombre_bodega',
       'tipo_y', 'region', 'pais', 'antiguedad_dias'],
      dtype='object')

In [30]:
# eliminar nulos en precio y cantidad
df_elasticidad = df_elasticidad.dropna(subset=["precio", "cantidad"])

# evitar ceros o negativos
df_elasticidad = df_elasticidad[(df_elasticidad["precio"] > 0) & (df_elasticidad["cantidad"] > 0)]

In [31]:
df_elasticidad["fecha"] = pd.to_datetime(df_elasticidad["fecha"])

In [32]:
df_elasticidad.groupby("id_ticket").agg({
    "cantidad": "sum",
    "precio": "mean"
})

,cantidad,precio
id_ticket,,
8100-2079,3124,1.690082e+05
8100-2081,33,1.133233e+06
8100-2082,11,1.995000e+04
8100-2083,66,1.972833e+05
8100-2086,1452,2.390123e+05
...,...,...
X898-994,1,3.299500e+05
X898-996,1,2.999500e+05
X898-997,1,3.989500e+05


In [33]:
#preview
df_elasticidad.head()
df_elasticidad.info()
df_elasticidad.describe()

<class 'pandas.core.frame.DataFrame'>
Index: 7604213 entries, 0 to 8023313
Data columns (total 31 columns):
 #   Column           Dtype         
---  ------           -----         
 0   fecha            datetime64[ns]
 1   id_producto      object        
 2   id_bodega        object        
 3   tipo_evento      object        
 4   cantidad         int64         
 5   cantidad_2       float64       
 6   precio           float64       
 7   precio_total     float64       
 8   costo            float64       
 9   col_extra_1      object        
 10  col_extra_2      int64         
 11  col_extra_3      object        
 12  id_ticket        object        
 13  desc_producto    object        
 14  talla            object        
 15  color            object        
 16  marca            object        
 17  genero           object        
 18  categoria        object        
 19  subcategoria     object        
 20  patron           object        
 21  largo            object        
 22 

,fecha,cantidad,cantidad_2,precio,precio_total,costo,col_extra_2,antiguedad_dias
count,7604213,7.604213e+06,7.604213e+06,7.604213e+06,7.604213e+06,7.604213e+06,7.604213e+06,4.363897e+06
mean,2025-06-09 17:43:38.309270784,1.055792e+01,1.754710e+00,2.638850e+05,2.594758e+05,6.983239e+04,5.053449e-02,1.740912e+02
min,2024-08-30 00:00:00,1.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,5.000000e+00
25%,2025-01-09 00:00:00,2.000000e+00,1.000000e+00,1.379000e+05,1.349500e+05,3.361500e+04,0.000000e+00,1.050000e+02
50%,2025-06-16 00:00:00,1.200000e+01,1.000000e+00,1.899500e+05,1.879000e+05,4.967800e+04,0.000000e+00,1.480000e+02
75%,2025-11-05 00:00:00,1.600000e+01,1.000000e+00,2.749500e+05,2.699500e+05,7.455200e+04,0.000000e+00,2.070000e+02
max,2026-03-29 00:00:00,1.600000e+01,1.499500e+05,5.079000e+09,5.079000e+09,2.115126e+06,5.000000e+00,6.070000e+02
std,NaN,5.908639e+00,1.090896e+02,9.682360e+06,9.682365e+06,7.347535e+04,5.001187e-01,1.047175e+02


In [34]:
df_elasticidad["precio"].describe()
df_elasticidad["cantidad"].describe()

count    7.604213e+06
mean     1.055792e+01
std      5.908639e+00
min      1.000000e+00
25%      2.000000e+00
50%      1.200000e+01
75%      1.600000e+01
max      1.600000e+01
Name: cantidad, dtype: float64

In [35]:
df_elasticidad.groupby("categoria")["precio"].std().sort_values(ascending=False)

categoria
BOTAS            2.179864e+07
CAMISA           1.730010e+07
TENIS            1.631973e+07
VESTIDO          1.346958e+07
LLAVEROS         1.263013e+07
                     ...     
TARJETERO        1.837707e+04
SPLASH           1.495869e+04
BALETAS          1.258234e+04
ANTIBACTERIAL    1.200223e+04
CREMA            9.958932e+02
Name: precio, Length: 61, dtype: float64

In [36]:
df_elasticidad.groupby("id_producto")["precio"].std().describe()

count    3.556400e+04
mean     4.339218e+05
std      1.121697e+07
min      0.000000e+00
25%      8.001597e+03
50%      2.254109e+04
75%      4.499704e+04
max      1.107543e+09
Name: precio, dtype: float64

In [37]:
#limpieza caso extremo: eliminar precios con desviación estándar muy alta

# percentiles por seguridad
def filtrar_outliers(grupo):
    q1 = grupo["precio"].quantile(0.25)
    q3 = grupo["precio"].quantile(0.75)
    iqr = q3 - q1
    
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    
    return grupo[(grupo["precio"] >= lower) & (grupo["precio"] <= upper)]

df_elasticidad = df_elasticidad.groupby("id_producto", group_keys=False).apply(filtrar_outliers)

C:\Users\haloj\AppData\Local\Temp\ipykernel_34528\3130365101.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_elasticidad = df_elasticidad.groupby("id_producto", group_keys=False).apply(filtrar_outliers)


In [38]:
#filtro para productos con std=0
std_precio = df_elasticidad.groupby("id_producto")["precio"].std()

productos_validos = std_precio[std_precio > 0].index

df_elasticidad = df_elasticidad[df_elasticidad["id_producto"].isin(productos_validos)]

## Definir nivel de agregación (decisión clave)

Vamos a usar:

- id_producto
- ciudad
- year_week

In [39]:
df_elasticidad["semana"] = df_elasticidad["fecha"].dt.isocalendar().week
df_elasticidad["anio"] = df_elasticidad["fecha"].dt.year
df_elasticidad["mes"] = df_elasticidad["fecha"].dt.month
df_elasticidad["year_week"] = df_elasticidad["anio"].astype(str) + "-" + df_elasticidad["semana"].astype(str)

In [40]:
df_elasticidad.head()

,fecha,id_producto,id_bodega,tipo_evento,cantidad,cantidad_2,precio,precio_total,costo,col_extra_1,...,ciudad,nombre_bodega,tipo_y,region,pais,antiguedad_dias,semana,anio,mes,year_week
0,2024-10-05,PROD_000000,STORE_000048,venta,1,1.0,923950.0,692962.5,325637.0,25,...,CITY_000006,Multimarca,STORE_TYPE_000013,STORE_DESC_000440,URBAN,221.0,40,2024,10,2024-40
1,2024-12-21,PROD_000000,STORE_000146,venta,1,1.0,970950.0,825307.5,325637.0,15,...,CITY_000014,Multimarca,STORE_TYPE_000010,STORE_DESC_000399,URBAN,111.0,51,2024,12,2024-51
2,2025-03-17,PROD_000000,STORE_000225,venta,1,1.0,970950.0,922402.5,448758.0,0,...,CITY_000006,Multimarca,STORE_TYPE_000013,STORE_DESC_000436,URBAN,190.0,12,2025,3,2025-12
3,2024-09-14,PROD_000000,STORE_000312,venta,1,1.0,923950.0,739160.0,325637.0,20,...,CITY_000006,Multimarca,STORE_TYPE_000010,STORE_DESC_000377,PREMIUM,100.0,37,2024,9,2024-37
4,2025-07-12,PROD_000000,STORE_000060,venta,1,1.0,970950.0,970950.0,448758.0,0,...,CITY_000014,Multimarca,STORE_TYPE_000010,STORE_DESC_000363,PREMIUM,154.0,28,2025,7,2025-28


In [41]:
df_agg = df_elasticidad.groupby([
    "id_producto",
    "ciudad",
    "year_week"
]).agg({
    "cantidad": "sum",        # demanda total
    "precio": "mean",         # precio promedio
    "costo": "mean",          # opcional para margen
    "categoria": "first",
    "marca": "first",
    "temporada": "first"
}).reset_index()

In [42]:
#filtro para evitar problemas en log
df_agg = df_agg[
    (df_agg["precio"] > 0) &
    (df_agg["cantidad"] > 0)
]
df_agg

,id_producto,ciudad,year_week,cantidad,precio,costo,categoria,marca,temporada
0,PROD_000000,CITY_000002,2026-7,12,929950.000000,429808.0,RELOJ,BRAND_000022,2026-0 (COMPRA Ene 01 - Mar 31)
1,PROD_000000,CITY_000002,2026-8,16,929950.000000,429808.0,RELOJ,BRAND_000022,2026-0 (COMPRA Ene 01 - Mar 31)
2,PROD_000000,CITY_000003,2024-36,12,923950.000000,325637.0,RELOJ,BRAND_000022,2026-0 (COMPRA Ene 01 - Mar 31)
3,PROD_000000,CITY_000003,2024-49,13,970950.000000,325637.0,RELOJ,BRAND_000022,2026-0 (COMPRA Ene 01 - Mar 31)
4,PROD_000000,CITY_000003,2024-51,1,970950.000000,325637.0,RELOJ,BRAND_000022,2026-0 (COMPRA Ene 01 - Mar 31)
...,...,...,...,...,...,...,...,...,...
422439,PROD_035577,CITY_000009,2026-10,2,125000.000000,0.0,None,None,None
422440,PROD_035577,CITY_000009,2026-13,17,209770.280909,0.0,None,None,None
422441,PROD_035577,CITY_000009,2026-4,10,222365.000000,0.0,None,None,None
422442,PROD_035577,CITY_000009,2026-5,2,270.745000,0.0,None,None,None


In [43]:
#logaritmo para elasticidad
df_agg["log_precio"] = np.log(df_agg["precio"])
df_agg["log_cantidad"] = np.log(df_agg["cantidad"])

In [44]:
#margen
df_agg["margen"] = df_agg["precio"] - df_agg["costo"]
#precio relativo 
precio_categoria = df_agg.groupby("categoria")["precio"].transform("mean")

df_agg["precio_relativo"] = df_agg["precio"] / precio_categoria
df_agg["log_precio_relativo"] = np.log(df_agg["precio_relativo"])

In [57]:

#separar variables numéricas
X_num = df_agg[[
    "log_precio"
]]
#dummies de categoricas
X_cat = pd.get_dummies(
    df_agg[['ciudad', 'categoria', 'marca', 'temporada']],
    drop_first=True
)
X = pd.concat([X_num, X_cat], axis=1)
X = X.astype(float)


X = sm.add_constant(X)

y = df_agg["log_cantidad"]

In [58]:
X.dtypes.value_counts()

float64    146
Name: count, dtype: int64

### Modelo base (OLS con statsmodels)

In [62]:
# modelo
import statsmodels.formula.api as smf
model = smf.ols(
    """
    log_cantidad ~ log_precio 
    + C(id_producto)
    + C(ciudad)
    + C(year_week)
    """,
    data=df_agg
).fit()

print(model.summary())

MemoryError: Unable to allocate 17.7 GiB for an array with shape (422444, 5629) and data type float64